In [ ]:
%pip install .

In [1]:
import base64
import io
import pickle
import functools

from IPython.display import HTML, display
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hnmf_tr_optimizer.hnmf_optimizer import HNMFOptimizer, NewHNMFOptimizer, flatten, unflatten
from hnmf_tr_optimizer.clusts import result_analysis

from dls_model import observational_matrix, gen_bounds, clustering_preprocess, InitParamsGenerator
# from dls_lognormal import g1, scatter_vector, diffusion_coef

# jax.config.update("jax_enable_x64", True)
# plt.style.use('Solarize_Light2')

| name       | 100 nm | 200 nm | 500 nm | 1000 nm | 1mM NaCl |
|-----------|--------|--------|--------|---------|----------|
| stock_100nm  | 1      | 0      | 0      | 0       | 0        |
| stock_200nm  | 0      | 1      | 0      | 0       | 0        |
| stock_500nm  | 0      | 0      | 1      | 0       | 0        |
| stock_1000nm | 0      | 0      | 0      | 1       | 0        |
| mix_1        | 0      | 0.8    | 0      | 0.2     | 0        |
| mix_2        | 0      | 0.8    | 0      | 0.2     | 0.25     |
| mix_3        | 0      | 0.8    | 0      | 0.2     | 3        |
| mix_4        | 0.3333 | 0.3333 | 0.3333 | 0       | 0        |
| mix_5        | 0.3333 | 0.3333 | 0.3333 | 0       | 0.25     |
| mix_6        | 0.3333 | 0.3333 | 0.3333 | 0       | 1        |
| mix_7        | 0.3333 | 0.3333 | 0.3333 | 0       | 3        |
| mix_8        | 0.3333 | 0.3333 | 0.3333 | 0       | 9        |
| mix_9        | 0.3333 | 0.3333 | 0.3333 | 0       | 0        |
| mix_10       | 0.25   | 0.25   | 0.25   | 0.25    | 0.25     |
| mix_11       | 0.25   | 0.25   | 0.25   | 0.25    | 1        |
| mix_12       | 0.25   | 0.25   | 0.25   | 0.25    | 3        |
| mix_13       | 0.25   | 0.25   | 0.25   | 0.25    | 9        |
| mix_14       | 0.25   | 0.25   | 0.25   | 0.25    | 19       |
| mix_15       | 0.25   | 0.25   | 0.25   | 0.25    | 0        |


In [2]:
### read in experimental data as well as real time values which have a custom pattern I don't wanna copy ###

df = pd.read_csv("~/repos/DLS/Experimental_data_083122/stock_100nm.csv", delimiter='\t', header=None)
df = df.iloc[1:] # remove strange first point
d = jnp.array(df.to_numpy())

t = d[:, 0] * 1e-3
theta = jnp.arange(30., 151, 5)

exp_obs = d[:, 1:].T

div = jax.vmap(lambda x: x/x[0])
norm_exp_obs = div(exp_obs)


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [3]:
# ### generate synthetic data ###

# exp = 2
# noise_level = 1e-9

# if exp == 1:
#     amp = jnp.array([1.])
#     mu = jnp.array([15.])
#     sig = jnp.array([4.])
# elif exp == 2:
#     amp = jnp.array([0.6, 0.4])
#     amp = amp/jnp.sum(amp)
#     mu = jnp.array([9., 19])
#     sig = jnp.array([1., 1.])
# elif exp == 3:
#     amp = jnp.array([0.6, 1, 0.4])
#     amp = amp/jnp.sum(amp)
#     mu = jnp.array([5., 15, 40])
#     sig = jnp.array([2., 4, 8])
# elif exp == 4:
#     amp = jnp.array([0.4, 0.8, 0.6, 1])
#     amp = amp/jnp.sum(amp)
#     mu = jnp.array([10., 20, 30, 40])
#     sig = jnp.array([2., 5, 3, 6])
# elif exp == 5:
#     # amp = jnp.array([0.2, 0.4, 0.6, 0.3, 0.4])
#     # amp = amp/jnp.sum(amp)
#     # mu = jnp.array([10., 20, 30, 45, 70])
#     # sig = jnp.array([1., 1.5, 3, 2, 5])
#     #### old ####
#     amp = jnp.array([0.2, 0.4, 0.6, 0.3, 0.4])
#     amp = amp/jnp.sum(amp)
#     mu = jnp.array([10., 20, 30, 40, 50])
#     sig = jnp.array([1., 4, 3, 2, 5])


# # Kbt = 4.11e-21 # Joules
# # vis = 0.00089 # viscosity of water at 25C
# l = 633e-9 # He-Ne laser wavelength
# n = 1.33 # water refractive index

# # theta = jnp.concatenate([jnp.arange(20., 50, 10), jnp.arange(50, 121, 5)]) # degrees
# q = (4*jnp.pi*n/l)*jnp.sin(jnp.radians(theta/2))

# # t = jnp.linspace(1e-7, 1e-3, 100)
# observations = observational_matrix(q, t, amp, mu, sig)


# key = jax.random.key(1337)
# key, subkey = jax.random.split(key)
# noise = jax.random.normal(subkey, shape=observations.shape)*noise_level

# observations = observations + noise


In [4]:
#### plotting functions ####


def plt2d(data, t):
    plt.clf()
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    ax.set_title('observations')
    ax.plot(t, data)
    ax.set_xscale('log')

    fig.tight_layout()

    plt.show()

def normal_distribution(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)
nnn = jax.vmap(normal_distribution, in_axes=(None, 0, 0, 0))

def plt_norms(amp, mu, sig):
    x_values = jnp.linspace(0, 25, 1000)
    real_distributions = nnn(x_values, amp, mu, sig)
    real_full_dist = jnp.sum(real_distributions, axis=0)
    plt.clf()
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=150)
    ax.plot(x_values, real_distributions.T, linestyle='--')
    ax.plot(x_values, real_full_dist)
    fig.tight_layout()
    plt.show()

def plotly_3d_surface(obs, pred_obs, x, y, title=None):
    # Initialize figure with 4 3D subplots
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}],])
    fig.add_trace(go.Surface(x=x, y=y, z=obs, coloraxis = "coloraxis1"),row=1, col=1)
    fig.add_trace(go.Surface(x=x, y=y, z=pred_obs, coloraxis = "coloraxis2"),row=1, col=2)
    fig.add_trace(go.Surface(x=x, y=y, z=obs-pred_obs, coloraxis = "coloraxis3"),row=1, col=3)
    fig.update_layout(
        title_text=title,
        coloraxis1=dict(
            colorscale="Viridis",
            colorbar=dict(title="Observations", x=0.25)
        ),
        coloraxis2=dict(
            colorscale="Viridis",
            colorbar=dict(title="Predictions", x=0.6)
        ),
        coloraxis3=dict(
            colorscale="Viridis",
            colorbar=dict(title="Residuals", x=0.95)
        )
    )
    return fig

plt3d = functools.partial(plotly_3d_surface, x=jnp.log10(t), y=theta)

def multi_3d_surface(obs, pred_obs, run_names, annotations, x, y, title=None):
    # Initialize figure with 4 3D subplots
    num_rows = len(obs)
    fig = make_subplots(
        rows=num_rows, cols=3,
        specs=num_rows*[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}],],
        # vertical_spacing=0.1
        )

    for i, (ob, pred, run_name, annotation) in enumerate(zip(obs, pred_obs, run_names, annotations)):
        # Choose a vertical center in "paper" coords for row i
        y_center = 1 - (i + 0.5)/num_rows
        # Make colorbars fill the vertical space for that row
        bar_len = 1.0/num_rows * 0.8
        
        # Common kwargs for colorbars so they line up for each row
        bar_kwargs = dict(
            y=y_center, 
            len=bar_len, 
            yref='paper', 
            lenmode='fraction'
        )

        fig.add_trace(go.Surface(x=x, y=y, z=ob, colorscale="Viridis", colorbar=dict(title="Observations", x=0.25, **bar_kwargs)),row=i+1, col=1)
        fig.add_trace(go.Surface(x=x, y=y, z=pred, colorscale="Viridis", colorbar=dict(title="Predictions", x=0.6, **bar_kwargs)),row=i+1, col=2)
        fig.add_trace(go.Surface(x=x, y=y, z=ob-pred, colorscale="Viridis", colorbar=dict(title="Residuals", x=0.95, **bar_kwargs)),row=i+1, col=3)
        # Add annotation to the left of that row
        y_top = 1 - (i+0.1)/num_rows
        fig.add_annotation(
            text=f"{run_name}\n{annotation}".replace("\n", "<br>"), 
            showarrow=False,
            yref="paper", 
            x=0.05,            # place it just to the left
            y=y_top,         # line up with the same row center
            font=dict(size=14),
            bordercolor="black",
            borderwidth=1,
            bgcolor="white",
            opacity=0.8
        )
    fig.update_layout(
        title_text=title,
        width=1400,  # Increase width
        height=800 + (300 * num_rows),  # Dynamically increase height based on number of rows
        margin=dict(l=10, r=10, t=50, b=10),  # Reduce unnecessary margins
        font=dict(size=14),
    )
    return fig

multi3d = functools.partial(multi_3d_surface, x=jnp.log10(t), y=theta)

In [5]:
# plt_norms(amp, mu, sig)
# plt2d(exp_obs.T, t)

In [6]:
### read in experimental data as well as real time values which have a custom pattern I don't wanna copy ###

def run(run_name, k):
    df = pd.read_csv(f"~/repos/DLS/Experimental_data_083122/{run_name}.csv", delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0] * 1e-3
    theta = jnp.arange(30., 151, 5)
    l = 633e-9 # He-Ne laser wavelength
    n = 1.33 # water refractive index

    # theta = jnp.concatenate([jnp.arange(20., 50, 10), jnp.arange(50, 121, 5)]) # degrees
    q = (4*jnp.pi*n/l)*jnp.sin(jnp.radians(theta/2))


    exp_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    norm_exp_obs = div(exp_obs)

    ### single optimization - filter within a time window and minimize ###

    seed = 1337
    sample_rate = 1.0

    # index = jnp.logical_and(t > 1e-5, t < 1e-2)
    index = jnp.logical_and(t > 0, t < 1e3)
    t_window = t[index]
    # observations_window = observations[:, index]
    # observations_window = exp_obs[:, index]
    observations_window = norm_exp_obs[:, index]

    opt = HNMFOptimizer(
        model_fn=observational_matrix,
        param_generator=InitParamsGenerator(),
        bound_generator=gen_bounds,
        input_args = ('q', 't'),
        param_args=('amp', 'mu', 'sig'),
        constants = {},
        min_k=1,
        max_k=1,
        nsim=100,
        sample_rate = sample_rate,
        sample_seed=seed
    )
    optimizer = opt.setup_optimizer(k, (q, t_window), observations_window, opt_options={
            'fatol': 1e-12,
            'frtol': 0,
            'maxiter': 2000,
            'gatol': 1e-10
        }
    )
    flat_init, _ = opt.flatten(*opt.param_generator(k))
    res = optimizer.full_trace_minimize(flat_init, opt.rand_key)
    print(f"finished minimization: {len(res)} steps")

    df = pd.DataFrame(res)
    df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(k)))
    step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])
    sol = step_params.iloc[-1]
    pred_observations = observational_matrix(q, t, sol['amp'], sol['mu'], sol['sig'])

    return sol, norm_exp_obs, pred_observations



In [7]:
obs, preds, sols = [],[],[]
run_names = [
    'stock_100nm',
    'stock_200nm',
    'stock_500nm',
    'stock_1000nm',
    'mix_1',
]

for r in run_names:
    sol, norm_exp_obs, pred_observations = run(r, 1)
    sols.append(sol)
    obs.append(norm_exp_obs)
    preds.append(pred_observations)

sol_strs = ['\n'.join(str(s.apply(lambda x: ", ".join([f"{xx:.4f}" for xx in x.tolist()]))).split('\n')[:-1]) for s in sols]

finished minimization: 36 steps
finished minimization: 37 steps
finished minimization: 37 steps
finished minimization: 45 steps
finished minimization: 40 steps


In [9]:
fff = multi3d(obs, preds, run_names, sol_strs)
with open('plots.html', 'w') as f:
    f.write(fff.to_html())

fff.show()

In [ ]:
### single optimization - filter within a time window and minimize ###

k = 1
seed = 1337
sample_rate = 1.0

# index = jnp.logical_and(t > 1e-5, t < 1e-2)
index = jnp.logical_and(t > 0, t < 1e3)
t_window = t[index]
# observations_window = observations[:, index]
# observations_window = exp_obs[:, index]
observations_window = norm_exp_obs[:, index]


opt = HNMFOptimizer(
    model_fn=observational_matrix,
    param_generator=InitParamsGenerator(),
    bound_generator=gen_bounds,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig'),
    constants = {},
    min_k=exp-1,
    max_k=exp+1,
    nsim=100,
    sample_rate = sample_rate,
    sample_seed=seed
)
optimizer = opt.setup_optimizer(k, (q, t_window), observations_window, opt_options={
        'fatol': 1e-12,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-10
    }
)
flat_init, _ = opt.flatten(*opt.param_generator(k))
res = optimizer.full_trace_minimize(flat_init, opt.rand_key)
print(f"finished minimization: {len(res)} steps")

df = pd.DataFrame(res)
df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(k)))
step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])
sol = step_params.iloc[-1]
pred_observations = observational_matrix(q, t, sol['amp'], sol['mu'], sol['sig'])

plt3d(norm_exp_obs, pred_observations, title='normalized experimental observations').show()

In [ ]:
### run 2 sources with a split boundary at the old mu ###

def modified_bounds(k, old_mu, old_ind):
    (lb_amp, lb_mu, lb_sig), (ub_amp, ub_mu, ub_sig) = gen_bounds(k)
    ub_mu[old_ind] = old_mu[old_ind]
    lb_mu[-1] = old_mu[old_ind]
    return (lb_amp, lb_mu, lb_sig), (ub_amp, ub_mu, ub_sig)


curr_bounds = functools.partial(modified_bounds, old_mu=sol['mu'], old_ind=0)


### single optimization - filter within a time window and minimize ###

k = 2
seed = 1337
sample_rate = 1.0

# index = jnp.logical_and(t > 1e-5, t < 1e-2)
index = jnp.logical_and(t > 0, t < 1e3)
t_window = t[index]
# observations_window = observations[:, index]
# observations_window = exp_obs[:, index]
observations_window = norm_exp_obs[:, index]


opt = HNMFOptimizer(
    model_fn=observational_matrix,
    param_generator=InitParamsGenerator(),
    bound_generator=curr_bounds, #### use new bounds for splitting #####
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig'),
    constants = {},
    min_k=exp-1,
    max_k=exp+1,
    nsim=100,
    sample_rate = sample_rate,
    sample_seed=seed
)
optimizer = opt.setup_optimizer(k, (q, t_window), observations_window, opt_options={
        'fatol': 1e-12,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-10
    }
)

# flat_init, _ = opt.flatten(*opt.param_generator(k))
old_amp, old_mu, old_sig = sol
old_ind = 0
amp = jnp.concat([old_amp, jnp.array([old_amp[old_ind]*0.3])])
amp = amp.at[old_ind].set(old_amp[old_ind]*0.7)

mu = jnp.concat([old_mu, jnp.array([old_mu[old_ind] + 0.6*old_sig[old_ind]])])
mu = mu.at[old_ind].set(old_mu[old_ind] - 0.1*old_sig[old_ind])

sig = jnp.concat([old_sig, jnp.array([old_sig[old_ind]/2])])
sig = sig.at[old_ind].set(old_sig[old_ind]*0.8)

flat_init, _ = opt.flatten(amp, mu, sig)

res2 = optimizer.full_trace_minimize(flat_init, opt.rand_key)
print(f"finished minimization: {len(res2)} steps")

df = pd.DataFrame(res2)
df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(k)))
step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])
sol2 = step_params.iloc[-1]
pred_observations = observational_matrix(q, t, sol2['amp'], sol2['mu'], sol2['sig'])

plt3d(norm_exp_obs, pred_observations, title='normalized experimental observations').show()


In [ ]:
amp = jnp.array([0.8, 0.2])
mu = jnp.array([0.2, 0.4])
sig = jnp.array([0.05, 0.5])

# pred_observations = observational_matrix(q, t, amp, mu, sig)
# plt3d(norm_exp_obs, pred_observations, title='normalized experimental observations').show()



flat_init, _ = opt.flatten(amp, mu, sig)

res2 = optimizer.full_trace_minimize(flat_init, opt.rand_key)
print(f"finished minimization: {len(res2)} steps")

df = pd.DataFrame(res2)
df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(k)))
step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])
sol2 = step_params.iloc[-1]
pred_observations = observational_matrix(q, t, sol2['amp'], sol2['mu'], sol2['sig'])

plt3d(norm_exp_obs, pred_observations, title='normalized experimental observations').show()


## Notes

#### notes with Petsev

inspect weird artifacts in residuals (fourier?)

for two peaks - try different distances between distributions and different amplitudes - plot distance vs amplitude ratio and define curve where they are separable

get access to cluster - center for advanced research computing

wavevector $q$ is a function of the angle $\theta$ is given by
$$q = \frac{4\pi n}{\lambda_0} \sin(\frac{\theta}{2})$$

Diffusion coefficient $D$ is a function of the size/radius $r$
$$D = \frac{k_B T}{6 \pi \eta r} = \frac{c}{r}$$

let $\Gamma = D q^2$ be the decay rate corresponding to the particular radius (let angle be fixed for now). We model the correlation function $g_1$ as the laplace transform of the decay $\Gamma$:

$$g_1(\tau) = \int_{0}^{\infty} G(\Gamma) e^{-\Gamma\tau} \,d\Gamma$$

Where $\tau$ is the time delay for the correlation function. We model the distribution of diffusion coefficients, $A(D)$, as a sum of normal distributions and then relate it to $g_1$:

$$A_i(D; \alpha_i, \mu_i, \sigma_i) = \frac{\alpha_i}{\sigma_i q^4\sqrt{2\pi}} \exp\left(-\frac{(D - \mu_i q^2)^2}{2\sigma_i^2 q^4}\right)$$

$$A(D; \alpha, \mu, \sigma) = \sum_{i=1}^k A_i(D)$$

$$g_1(\tau) = \int_{0}^{\infty} A(D) e^{-q^2 D\tau} \,dD$$

We can express this in the closed-form

$$
g_1(t) = \sum_{i=1}^{k} \left(\frac{\alpha_i}{2} \right) \exp\left(-\frac{\mu_i^2}{2\sigma_i^2}\right) \exp\left(\left( \frac{\frac{\mu_i}{\sigma_i} + q^2 \sigma_i t}{\sqrt{2}} \right)^2\right) \operatorname{erfc} \left( \frac{\frac{\mu_i}{\sigma_i} + q^2 \sigma_i t}{\sqrt{2}} \right)
$$

In [118]:
### g1 formula for normally distributed radii ###
def scatter_vector(theta, lambda_0=633e-9, n=1.33):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=138.e-23, T=298.15, eta=0.00089, n=1.33):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    n - refractive index of water
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def g1_single(mu, sigma, alpha, theta, tau):
    """
    Parameters:
      tau      - time lag (scalar or array)
      alpha    - amplitude scaling factor
      mu       - mean particle radius (m)
      sigma    - standard deviation of the radius (m)
      theta    - scattering angle (radians)
      lambda_0 - laser wavelength (m, default 633e-9)
      n        - refractive index of the medium (default 1.33)
      k_B      - Boltzmann constant (default 138e-23 J/K)
      T        - temperature (default 298.15 K)
      eta      - viscosity (default 0.00089 Pa*s)

    Returns:
      g1: the computed field autocorrelation function.
    """
    q = scatter_vector(theta)
    D_mu = diffusion_coef(mu)
    A = q**2 * mu * D_mu * tau
    exp_term = jnp.exp(-A / mu)
    erfc_arg1 = (mu - jnp.sqrt(A)) / (jnp.sqrt(2) * sigma)
    erfc_arg2 = (mu + jnp.sqrt(A)) / (jnp.sqrt(2) * sigma)
    return (alpha / 2) * exp_term * (jax.scipy.special.erfc(erfc_arg1) + jax.scipy.special.erfc(erfc_arg2))


g1_time = jax.vmap(
    g1_single,
    in_axes = (None, None, None, None, 0)
)

g1_k = jax.vmap(
    g1_single,
    in_axes=(0, 0, 0, None, None)
)

def g1_k_sum(r, sig, a, q, tau):
    return jnp.sum(g1_k(r, sig, a, q, tau), axis=0) # shape (k, t) -> (t,)

g1_theta = jax.vmap(
    g1_k_sum,
    in_axes = (None, None, None, 0, None)
)

# def g1(r, sig, a, theta, tau):
def g1(theta, tau, a, r, sig):
    """
    r - radius/mean size
    sig - std of radius/size
    a - amplitude of size distribution
    theta - angle in radians
    tau - time in seconds (milliseconds?)
    """
    # shitty wrapper to make inspect.getfullargspec work within HNMFOptimizer
    return g1_theta(r, sig, a, theta, tau)
